In [47]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from datetime import datetime
from scipy.stats import pearsonr
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
import pingouin as pg
import warnings

# Suppress specific warnings that we're fixing
warnings.filterwarnings("ignore", message="DataFrame.groupby with axis=1 is deprecated")
warnings.filterwarnings("ignore", message="DataFrameGroupBy.diff with axis=1 is deprecated")
warnings.filterwarnings("ignore", message="The MLE may be on the boundary of the parameter space")
warnings.filterwarnings("ignore", message="The default of observed=False is deprecated")

# ==============================================================================
# CONFIGURATION AND DATA STRUCTURES
# ==============================================================================

@dataclass
class AnalysisConfig:
    """Centralized configuration for all analysis parameters."""
    base_output_dir: Path = Path('motor_learning_output')
    min_complete_strides: int = 20
    motor_noise_strides: int = 20
    motor_noise_threshold: float = 0.3
    success_rate_threshold: float = 0.68
    target_size_threshold: float = 0.31
    max_strides_threshold: int = 415
    figure_dpi: int = 300
    alpha_level: float = 0.05
    age_bins: List[int] = field(default_factory=lambda: [7, 10, 13, 16, 18])
    age_labels: List[str] = field(default_factory=lambda: ['7-10', '10-13', '13-16', '16-18'])
    trial_type_mapping: Dict[str, str] = field(default_factory=lambda: {
        'primer': 'vis1', 'trial': 'invis', 'vis': 'vis2', 'pref': 'pref'
    })
    
    def __post_init__(self):
        # Ensure base_output_dir is a Path object
        if not isinstance(self.base_output_dir, Path):
            self.base_output_dir = Path(self.base_output_dir)
        
        # Create all output directories
        dirs = ['figures', 'individual_plots', 'population_plots', 'statistical_plots', 
                'reports', 'exports', 'processed_data']
        
        for d in dirs:
            # Create proper attribute names and paths
            attr_name = f'{d.replace("_", "")}dir'
            dir_path = self.base_output_dir / d
            setattr(self, attr_name, dir_path)
            dir_path.mkdir(parents=True, exist_ok=True)
        
        # Create figure subdirectories
        for subdir in ['individual_plots', 'population_plots', 'statistical_plots']:
            (self.base_output_dir / 'figures' / subdir).mkdir(parents=True, exist_ok=True)
        
        # Set the processed data file path
        self.processed_data_file = self.processeddatadir / 'processed_data.pkl'


@dataclass
class SubjectData:
    subject_id: str
    metadata: Dict[str, Any]
    trial_data: Dict[str, Dict[str, Any]]
    
    @property
    def age(self) -> float:
        return self.metadata.get('age_months', np.nan) / 12

# ==============================================================================
# BASE CLASSES
# ==============================================================================

class BaseProcessor(ABC):
    def __init__(self, config: AnalysisConfig, debug: bool = True):
        self.config = config
        self.debug = debug
    
    def log(self, message: str, level: str = "info"):
        if self.debug:
            symbols = {"info": "📊", "warning": "⚠️", "error": "❌", "success": "✓"}
            print(f"{symbols.get(level, '•')} {message}")

class BaseVisualizer(ABC):
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.colors = {
            'primary': '#667eea', 'secondary': '#764ba2', 'success': '#28a745',
            'warning': '#ffc107', 'danger': '#dc3545', 'vis1': '#1f77b4',
            'invis': '#ff7f0e', 'vis2': '#2ca02c'
        }
    
    def save_figure(self, fig: plt.Figure, filename: str, subdir: str = 'general'):
        """Save figure using config paths and print the save location"""
        subdir_map = {
            'individual': self.config.base_output_dir / 'figures' / 'individual_plots',
            'population': self.config.base_output_dir / 'figures' / 'population_plots',
            'statistical': self.config.base_output_dir / 'figures' / 'statistical_plots'
        }
        
        # Use config.figuresdir as fallback and ensure it exists
        save_path = subdir_map.get(subdir, self.config.figuresdir) / filename
        save_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
        
        # Use config.figure_dpi setting
        fig.savefig(save_path, dpi=self.config.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        
        # Print the save location like reports do
        print(f"Figure saved to: {save_path}")
        return save_path
    
    def add_trendline(self, ax, x, y):
        x_clean, y_clean = pd.to_numeric(x, errors='coerce'), pd.to_numeric(y, errors='coerce')
        valid = x_clean.notna() & y_clean.notna()
        if valid.sum() < 2:
            return
        
        x_vals, y_vals = x_clean[valid], y_clean[valid]
        try:
            coeffs = np.polyfit(x_vals, y_vals, 1)
            trendline = np.poly1d(coeffs)
            r, p = pearsonr(x_vals, y_vals)
            ax.plot(x_vals, trendline(x_vals), 'r--', alpha=0.8, linewidth=2)
            ax.text(0.05, 0.95, f'r² = {r**2:.3f}\np = {p:.3f}\nn = {len(x_vals)}', 
                   transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top', fontsize=10)
        except Exception:
            pass

# ==============================================================================
# DATA PROCESSING COMPONENTS
# ==============================================================================

class DataValidator:
    @staticmethod
    def validate_dataframe(df: pd.DataFrame, required_cols: List[str] = None) -> bool:
        return df is not None and not df.empty and (not required_cols or all(col in df.columns for col in required_cols))
    
    @staticmethod
    def detect_anomalies(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        if df is None or df.empty:
            return df, {}
        
        df = df.copy()
        df['Anomalous'] = False
        anomalies = {}
        
        # Time-based anomalies
        time_col = next((col for col in ['Time', 'Timestamp', 'Time (s)'] if col in df.columns), None)
        if time_col:
            df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
            time_diff = df[time_col].diff()
            jump_mask = time_diff > time_diff.quantile(0.99) * 5
            for idx in df.index[jump_mask.fillna(False)]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('time_jump')
        
        # Sum of gains and steps anomalies
        if 'Sum of gains and steps' in df.columns:
            high_mask = df['Sum of gains and steps'] > 4
            zero_mask = df['Sum of gains and steps'] == 0
            for idx in df.index[high_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_high')
            for idx in df.index[zero_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_zero')
        
        return df, anomalies

class FileLoader:
    @staticmethod
    def load_file(file_path: Path) -> Optional[pd.DataFrame]:
        try:
            df = pd.read_csv(file_path, sep='\t')
            if 'Stride Number' in df.columns:
                df['Stride Number'] = pd.to_numeric(df['Stride Number'], errors='coerce')
                df = df.dropna(subset=['Stride Number']).drop_duplicates(subset=['Stride Number']).sort_values('Stride Number')
            return df if not df.empty else None
        except Exception:
            return None

class TrialProcessor(BaseProcessor):
    def process_trial_files(self, subject_dir: Path, trial_prefix: str) -> Optional[pd.DataFrame]:
        all_files = sorted(subject_dir.glob(f"{trial_prefix}*.txt"))
        if not all_files:
            return None
        
        if trial_prefix == 'pref':
            return FileLoader.load_file(max(all_files, key=lambda f: f.stat().st_size))
        
        if len(all_files) == 1:
            return FileLoader.load_file(all_files[0])
        
        return self._combine_trial_fragments(all_files)
    
    def _combine_trial_fragments(self, files: List[Path]) -> Optional[pd.DataFrame]:
        dfs = [FileLoader.load_file(f) for f in files if FileLoader.load_file(f) is not None]
        if not dfs:
            return None
        
        combined = pd.concat(dfs, ignore_index=True)
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number').drop_duplicates('Stride Number')
        return combined

# ==============================================================================
# METRICS CALCULATION
# ==============================================================================

class MetricsEngine:
    def __init__(self, config: AnalysisConfig):
        self.config = config
    
    def calculate_period_metrics(self, period_data: pd.DataFrame, trial_type: str, condition: str) -> Dict:
        metrics = {f'{trial_type}_sr_{condition}_const': period_data['Success'].mean()}
        
        if 'Sum of gains and steps' in period_data.columns:
            sogs = period_data['Sum of gains and steps']
            metrics.update({
                f'{trial_type}_sd_{condition}_const': sogs.std(),
                f'{trial_type}_msl_{condition}_const': sogs.mean()
            })
            if 'Constant' in period_data.columns:
                metrics[f'{trial_type}_error_{condition}_const'] = (sogs - period_data['Constant']).mean()
        
        if all(col in period_data.columns for col in ['Right step length', 'Left step length']):
            asymmetry = self._calculate_asymmetry(period_data['Right step length'], period_data['Left step length'])
            if asymmetry is not None:
                metrics[f'{trial_type}_asymmetry_{condition}_const'] = asymmetry
        
        strides_between = self._calculate_strides_between_successes(period_data)
        if strides_between is not None:
            metrics[f'{trial_type}_strides_between_success_{condition}_const'] = strides_between
        
        return metrics
    
    def calculate_preference_metrics(self, pref_df: pd.DataFrame) -> Dict:
        metrics = {'mot_noise': None, 'pref_asymmetry': None}
        
        if pref_df is None or pref_df.empty or not all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
            return metrics
        
        right_steps = pref_df['Right step length']
        left_steps = pref_df['Left step length']
        right_clean = right_steps[(right_steps != 0) & (right_steps.notna())]
        left_clean = left_steps[(left_steps != 0) & (left_steps.notna())]
        
        min_required = self.config.motor_noise_strides
        if len(right_clean) < min_required or len(left_clean) < min_required:
            return metrics
        
        final_right, final_left = right_clean.iloc[-1], left_clean.iloc[-1]
        if final_right <= 0 or final_left <= 0:
            return metrics
        
        norm_right, norm_left = right_clean / final_right, left_clean / final_left
        min_length = min(len(norm_right), len(norm_left))
        if min_length < min_required:
            return metrics
        
        sum_steps = norm_right.iloc[:min_length] + norm_left.iloc[:min_length]
        
        if len(sum_steps) >= self.config.motor_noise_strides:
            noise = sum_steps.tail(self.config.motor_noise_strides).std()
            if not pd.isna(noise) and noise > 0:
                metrics['mot_noise'] = noise
        
        if len(right_clean) >= self.config.motor_noise_strides and len(left_clean) >= self.config.motor_noise_strides:
            last_n_right = right_clean.tail(self.config.motor_noise_strides) / final_right
            last_n_left = left_clean.tail(self.config.motor_noise_strides) / final_left
            asymmetry = self._calculate_asymmetry(last_n_right.values, last_n_left.values)
            if asymmetry is not None:
                metrics['pref_asymmetry'] = asymmetry
        
        return metrics
    
    def _calculate_asymmetry(self, right_values, left_values) -> Optional[float]:
        denominator = right_values + left_values
        valid_mask = denominator != 0
        if not valid_mask.any():
            return None
        asymmetry_vals = np.abs((right_values - left_values) / denominator)[valid_mask]
        return np.mean(asymmetry_vals) if len(asymmetry_vals) > 0 else None
    
    def _calculate_strides_between_successes(self, df: pd.DataFrame) -> Optional[float]:
        if df is None or 'Success' not in df.columns:
            return None
        df = df.reset_index(drop=True)
        success_positions = df.index[df['Success'] == 1].tolist()
        return np.mean(np.diff(success_positions)) if len(success_positions) >= 2 else None

# ==============================================================================
# DATA MANAGEMENT
# ==============================================================================

class DataManager(BaseProcessor):
    def __init__(self, metadata_path: str, data_root_dir: str, config: AnalysisConfig, 
                 force_reprocess: bool = False, debug: bool = True):
        super().__init__(config, debug)
        self.metadata_path = metadata_path
        self.data_root_dir = Path(data_root_dir)
        self.trial_processor = TrialProcessor(config, debug)
        self.metrics_engine = MetricsEngine(config)
        self.subjects: Dict[str, SubjectData] = {}
        self.metadata: pd.DataFrame = None
        
        if not force_reprocess and config.processed_data_file.exists():
            self._load_processed_data()
        else:
            self._process_all_data()
            self._save_processed_data()
    
    def _load_processed_data(self):
        try:
            with open(self.config.processed_data_file, 'rb') as f:
                saved_data = pickle.load(f)
            
            for subject_id, data in saved_data.items():
                self.subjects[subject_id] = SubjectData(
                    subject_id=subject_id, metadata=data['metadata'], trial_data=data['trial_data']
                )
            
            self.metadata = pd.DataFrame.from_dict(
                {subj: data.metadata for subj, data in self.subjects.items()}, orient='index'
            )
            self.log(f"Loaded {len(self.subjects)} subjects from cache", "success")
        except Exception as e:
            self.log(f"Failed to load cached data: {e}", "warning")
            self._process_all_data()
            self._save_processed_data()
    
    def _save_processed_data(self):
        try:
            save_data = {
                subject_id: {'metadata': subject.metadata, 'trial_data': subject.trial_data}
                for subject_id, subject in self.subjects.items()
            }
            with open(self.config.processed_data_file, 'wb') as f:
                pickle.dump(save_data, f)
            self.log(f"Saved processed data to {self.config.processed_data_file}", "success")
        except Exception as e:
            self.log(f"Failed to save processed data: {e}", "error")
    
    def _process_all_data(self):
        self._load_metadata()
        total_subjects = len(self.metadata)
        self.log(f"Processing {total_subjects} subjects...")
        
        for i, (_, row) in enumerate(self.metadata.iterrows(), 1):
            if self.debug and i % 10 == 0:
                self.log(f"Progress: {i}/{total_subjects}")
            
            subject_data = self._process_subject(row['ID'], row)
            if subject_data:
                self.subjects[row['ID']] = subject_data
    
    def _load_metadata(self):
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata['DOB'] = pd.to_datetime(self.metadata['DOB'], errors='coerce')
        self.metadata['Session Date'] = pd.to_datetime(self.metadata['Session Date'], errors='coerce')
        self.metadata = self.metadata.dropna(subset=['ID', 'age_months'])
    
    def _process_subject(self, subject_id: str, metadata_row: pd.Series) -> Optional[SubjectData]:
        subject_dir = self.data_root_dir / subject_id
        if not subject_dir.exists():
            return None
        
        trial_data = {}
        for original_type, mapped_type in self.config.trial_type_mapping.items():
            df = self.trial_processor.process_trial_files(subject_dir, original_type)
            if df is not None:
                if original_type != 'pref':
                    df, anomalies = self._process_trial_data(df)
                else:
                    df = df.drop_duplicates(subset='Left heel strike', keep='last')
                    anomalies = {}
                trial_data[mapped_type] = {'data': df, 'anomalies': anomalies}
        
        return SubjectData(subject_id=subject_id, metadata=metadata_row.to_dict(), trial_data=trial_data) if trial_data else None
    
    def _process_trial_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        if df is None or df.empty:
            return None, {}
        
        required_cols = ['Stride Number', 'Success', 'Upper bound success', 'Lower bound success', 'Constant']
        if not all(col in df.columns for col in required_cols):
            return None, {}
        
        df = df.sort_values('Stride Number')
        df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        df = df.drop_duplicates(subset='Stride Number', keep='last')
        
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        return DataValidator.detect_anomalies(df)
    
    def calculate_metrics(self) -> pd.DataFrame:
        results = []
        self.log(f"Calculating metrics for {len(self.subjects)} subjects...")
        
        for subject_id, subject in self.subjects.items():
            result = self._calculate_subject_metrics(subject)
            if result:
                results.append(result)
        
        if not results:
            self.log("No valid metrics calculated!", level="error")
            return pd.DataFrame()
        
        df = pd.DataFrame(results).infer_objects()
        
        # Save metrics to disk
        self._save_metrics(df)
        
        self.log(f"Successfully calculated metrics for {len(df)} subjects", level="success")
        return df
    
    def _save_metrics(self, metrics_df: pd.DataFrame):
        """Save calculated metrics to disk"""
        try:
            # Save as CSV for easy viewing
            csv_path = self.config.processeddatadir / 'calculated_metrics.csv'
            metrics_df.to_csv(csv_path, index=False)
            
            # Save as pickle for exact preservation
            pickle_path = self.config.processeddatadir / 'calculated_metrics.pkl'
            metrics_df.to_pickle(pickle_path)
            
            # Save as Excel for stakeholders
            excel_path = self.config.exportsdir / 'calculated_metrics.xlsx'
            metrics_df.to_excel(excel_path, index=False, engine='openpyxl')
            
            self.log(f"Metrics saved to CSV: {csv_path}", "success")
            self.log(f"Metrics saved to pickle: {pickle_path}", "success") 
            self.log(f"Metrics saved to Excel: {excel_path}", "success")
            
        except Exception as e:
            self.log(f"Failed to save metrics: {e}", "error")
    
    def load_metrics(self) -> Optional[pd.DataFrame]:
        """Load previously calculated metrics from disk"""
        pickle_path = self.config.processeddatadir / 'calculated_metrics.pkl'
        
        if pickle_path.exists():
            try:
                df = pd.read_pickle(pickle_path)
                self.log(f"Loaded metrics for {len(df)} subjects from {pickle_path}", "success")
                return df
            except Exception as e:
                self.log(f"Failed to load metrics: {e}", "error")
                return None
        else:
            self.log("No saved metrics found", "warning")
            return None
    
    def _calculate_subject_metrics(self, subject: SubjectData) -> Optional[Dict]:
        result = {
            'ID': subject.subject_id,
            'age': subject.age,
            'session_date': subject.metadata.get('Session Date')
        }
        
        for trial_type in ['vis1', 'invis', 'vis2']:
            trial_dict = subject.trial_data.get(trial_type)
            if not trial_dict or trial_dict['data'] is None or trial_dict['data'].empty or 'Success' not in trial_dict['data'].columns:
                continue
            
            df = trial_dict['data']
            
            for condition in ['max', 'min']:
                period_data, indices = self._get_period_data(df, condition)
                if period_data is not None and not period_data.empty:
                    metrics = self.metrics_engine.calculate_period_metrics(period_data, trial_type, condition)
                    result.update(metrics)
                    result[f'{trial_type}_{condition}_const_indices'] = indices
            
            if df is not None:
                result.update({
                    f'{trial_type}_min_target_size': df['Target size'].min() if 'Target size' in df.columns else None,
                    f'{trial_type}_max_constant': df['Constant'].max() if 'Constant' in df.columns else None,
                    f'{trial_type}_min_constant': df['Constant'].min() if 'Constant' in df.columns else None
                })
                
                if trial_type == 'invis':
                    result.update(self._calculate_condition_order(df))
        
        pref_dict = subject.trial_data.get('pref')
        if pref_dict:
            pref_metrics = self.metrics_engine.calculate_preference_metrics(pref_dict['data'])
            result.update(pref_metrics)
        
        return result
    
    def _get_period_data(self, df: pd.DataFrame, condition: str, length: int = 20) -> Tuple[Optional[pd.DataFrame], Optional[List]]:
        if df is None or df.empty or not all(col in df.columns for col in ['Target size', 'Constant']):
            return None, None
        
        min_target = df['Target size'].min()
        min_target_periods = df[df['Target size'] <= min_target + 0.001]
        if min_target_periods.empty:
            return None, None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max' 
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)]
        return (period_data.tail(length), period_data.index.tolist()) if not period_data.empty else (None, None)
    
    def _calculate_condition_order(self, df: pd.DataFrame) -> Dict:
        all_max_indices = df.index[df['Constant'] == df['Constant'].max()].tolist()
        all_min_indices = df.index[df['Constant'] == df['Constant'].min()].tolist()
        
        if all_max_indices and all_min_indices:
            first_max, first_min = min(all_max_indices), min(all_min_indices)
            return {'invis_max_first': first_max < first_min, 'invis_min_first': first_min < first_max}
        return {'invis_max_first': False, 'invis_min_first': False}
    
    def filter_subjects(self, max_target_size: float = None, min_age: float = None, 
                       max_age: float = None, required_trial_types: List[str] = None) -> 'DataManager':
        filtered_subjects = {}
        
        for subject_id, subject in self.subjects.items():
            age = subject.age
            if (min_age is not None and age < min_age) or (max_age is not None and age > max_age):
                continue
            
            if required_trial_types:
                missing_trials = [t for t in required_trial_types 
                                if t not in subject.trial_data or subject.trial_data[t]['data'] is None]
                if missing_trials:
                    continue
            
            valid = True
            for trial_type, trial_dict in subject.trial_data.items():
                if trial_dict and trial_dict['data'] is not None:
                    df = trial_dict['data']
                    if (max_target_size is not None and 'Target size' in df.columns and 
                        df['Target size'].min() > max_target_size):
                        valid = False
                        break
            
            if valid:
                filtered_subjects[subject_id] = subject
        
        new_manager = DataManager.__new__(DataManager)
        for attr in ['config', 'metadata_path', 'data_root_dir', 'debug', 'trial_processor', 'metrics_engine']:
            setattr(new_manager, attr, getattr(self, attr))
        new_manager.subjects = filtered_subjects
        new_manager.metadata = pd.DataFrame.from_dict(
            {subj: data.metadata for subj, data in filtered_subjects.items()}, orient='index'
        )
        return new_manager

# ==============================================================================
# STATISTICAL ANALYSIS - FIXED WARNINGS
# ==============================================================================

class StatisticalAnalyzer:
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        self.config = config
        self.metrics_df = metrics_df
        self.filtered_df = (metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold] 
                           if 'mot_noise' in metrics_df.columns else metrics_df)

    def _run_repeated_measures_anova_generic(self, metric_type: str) -> Dict:
        """Generic ANOVA runner for different metrics (sr, msl, sd)"""
        metric_names = {'sr': 'success_rate', 'msl': 'mean_stride_length', 'sd': 'stride_length_variability'}
        dv_name = metric_names[metric_type]
        
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if f'_{metric_type}_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type, condition = parts[0], parts[2]
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id, 'trial_type': trial_type, 'condition': condition,
                                dv_name: row[col], 'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna()
        if df_long.empty:
            return {'error': f'No valid {dv_name} data for analysis'}
        
        results = self._run_basic_anova(df_long, dv_name)
        results.update(self._run_covariate_analysis(df_long, dv_name))
        results['descriptive_stats'] = self._get_descriptive_stats(df_long, dv_name)
        return results
    
    def _run_basic_anova(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Run basic repeated measures ANOVA"""
        results = {}
        try:
            # Ensure data is properly formatted for pingouin
            df_long = df_long.copy()
            df_long['subject'] = df_long['subject'].astype(str)
            df_long['trial_type'] = df_long['trial_type'].astype('category')
            df_long['condition'] = df_long['condition'].astype('category')
            
            # Main effects and interaction
            for effect, within_vars in [('trial_type', 'trial_type'), ('condition', 'condition'), 
                                       ('interaction', ['trial_type', 'condition'])]:
                if (effect != 'interaction' and len(df_long[within_vars].unique()) > 1) or \
                   (effect == 'interaction' and len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1):
                    
                    # Use more robust ANOVA settings
                    aov = pg.rm_anova(data=df_long, dv=dv_name, within=within_vars, 
                                     subject='subject', detailed=True, effsize='ng2')
                    
                    if effect == 'interaction':
                        interaction_row = aov[aov['Source'].str.contains('trial_type \\* condition')]
                        if not interaction_row.empty:
                            results[f'{effect}_effect'] = self._extract_anova_results(interaction_row.iloc[0])
                    else:
                        results[f'{effect}_effect'] = self._extract_anova_results(aov.iloc[0])
        except Exception as e:
            results['basic_anova_error'] = str(e)
        return results
    
    def _extract_anova_results(self, anova_row) -> Dict:
        """Extract standardized results from ANOVA row"""
        return {
            'F': float(anova_row['F']),
            'p_value': float(anova_row['p-unc']),
            'effect_size': float(anova_row['ng2'])
        }
    
    def _run_covariate_analysis(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Run covariate analysis"""
        results = {}
        try:
            available_covariates = [cov for cov in ['age', 'motor_noise', 'pref_asymmetry'] 
                                  if cov in df_long.columns and df_long[cov].notna().sum() > 0]
            
            if available_covariates:
                results['ancova_covariates'] = available_covariates
                
                # Covariate correlations
                for cov in available_covariates:
                    cov_corr = df_long.groupby('subject').agg({dv_name: 'mean', cov: 'first'}).reset_index()
                    if len(cov_corr) > 5:
                        r, p = pearsonr(cov_corr[cov], cov_corr[dv_name])
                        results[f'{cov}_covariate_effect'] = {
                            'correlation': float(r), 'p_value': float(p),
                            'interpretation': f'{cov} effect on {dv_name}'
                        }
                
                # Mixed effects analysis
                try:
                    import statsmodels.api as sm
                    from statsmodels.formula.api import mixedlm
                    
                    covariate_terms = ' + '.join(available_covariates)
                    formula = f"{dv_name} ~ trial_type * condition + {covariate_terms}"
                    
                    # Improve convergence by using REML and better starting values
                    model = mixedlm(formula, df_long, groups=df_long['subject'], 
                                   re_formula="1")  # Simple random intercept
                    
                    # Fit with more robust settings
                    try:
                        fitted_model = model.fit(reml=True, maxiter=200)
                    except:
                        # Fallback: try with simpler model if convergence fails
                        simple_formula = f"{dv_name} ~ trial_type + condition + {covariate_terms}"
                        model = mixedlm(simple_formula, df_long, groups=df_long['subject'])
                        fitted_model = model.fit(reml=True, maxiter=100)
                    
                    covariate_effects = {}
                    for cov in available_covariates:
                        if cov in fitted_model.params.index:
                            covariate_effects[f'{cov}_effect'] = {
                                'coefficient': float(fitted_model.params[cov]),
                                'p_value': float(fitted_model.pvalues[cov]),
                                'confidence_interval': [float(fitted_model.conf_int().loc[cov, 0]), 
                                                      float(fitted_model.conf_int().loc[cov, 1])],
                                'interpretation': f'Change in {dv_name} per unit increase in {cov}'
                            }
                    results['mixed_effects_covariates'] = covariate_effects
                    
                except Exception as e:
                    results['mixed_effects_error'] = str(e)
            
        except Exception as e:
            results['ancova_error'] = str(e)
        return results
    
    def _get_descriptive_stats(self, df_long: pd.DataFrame, dv_name: str) -> Dict:
        """Get descriptive statistics"""
        return {
            'n_subjects': len(df_long['subject'].unique()),
            'n_observations': len(df_long),
            f'{dv_name}_overall': float(df_long[dv_name].mean()),
            f'std_{dv_name}_overall': float(df_long[dv_name].std()),
            'by_trial_type': df_long.groupby('trial_type')[dv_name].agg(['mean', 'std', 'count']).to_dict(),
            'by_condition': df_long.groupby('condition')[dv_name].agg(['mean', 'std', 'count']).to_dict()
        }

    def run_repeated_measures_anova(self) -> Dict:
        """Run repeated measures ANOVA for success rates"""
        return self._run_repeated_measures_anova_generic('sr')
    
    def run_repeated_measures_anova_msl(self) -> Dict:
        """Run repeated measures ANOVA for mean stride length"""
        return self._run_repeated_measures_anova_generic('msl')
    
    def run_repeated_measures_anova_sd(self) -> Dict:
        """Run repeated measures ANOVA for stride length variability"""
        return self._run_repeated_measures_anova_generic('sd')

    def run_age_stratified_anova(self, age_groups: Dict[str, List[float]] = None, 
                                min_subjects_per_group: int = 1) -> Dict:
        """Run ANOVA analysis stratified by age groups - FIXED WARNINGS"""
        if age_groups is None:
            age_groups = {'younger': [7, 12], 'middle': [12, 15], 'older': [15, 18]}
        
        # Prepare long-format data
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id, age = row.name, row.get('age', np.nan)
            if pd.isna(age):
                continue
                
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type, condition = parts[0], parts[2]
                        if pd.notna(row[col]):
                            long_rows.append({
                                'subject': subject_id, 'trial_type': trial_type, 'condition': condition,
                                'success_rate': row[col], 'age': age,
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['age', 'success_rate'])
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {
            'age_group_definitions': age_groups,
            'min_subjects_threshold': min_subjects_per_group,
            'total_subjects_analyzed': len(df_long['subject'].unique()),
            'age_range': [float(df_long['age'].min()), float(df_long['age'].max())],
            'group_analyses': {}, 'between_group_comparisons': {}, 'summary_comparison': {}
        }
        
        # Assign subjects to age groups
        df_long['age_group'] = None
        group_assignments = {}
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_long['age'] >= min_age) & (df_long['age'] < max_age)
            df_long.loc[mask, 'age_group'] = group_name
            
            subjects_in_group = df_long[mask]['subject'].unique()
            group_assignments[group_name] = {
                'subjects': list(subjects_in_group), 'n_subjects': len(subjects_in_group),
                'age_range': [float(df_long[mask]['age'].min()) if len(subjects_in_group) > 0 else np.nan,
                             float(df_long[mask]['age'].max()) if len(subjects_in_group) > 0 else np.nan],
                'mean_age': float(df_long[mask]['age'].mean()) if len(subjects_in_group) > 0 else np.nan
            }
        
        results['group_assignments'] = group_assignments
        
        # Run ANOVA for each age group
        for group_name, group_info in group_assignments.items():
            if group_info['n_subjects'] < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={group_info["n_subjects"]}, minimum={min_subjects_per_group})'
                }
                continue
            
            group_data = df_long[df_long['age_group'] == group_name].copy()
            group_results = {}
            
            # Ensure proper data types
            group_data['subject'] = group_data['subject'].astype(str)
            group_data['trial_type'] = group_data['trial_type'].astype('category')
            group_data['condition'] = group_data['condition'].astype('category')
            
            try:
                aov_trial = pg.rm_anova(data=group_data, dv='success_rate', 
                                       within='trial_type', subject='subject', 
                                       detailed=True, effsize='ng2')
                group_results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size_eta2': float(aov_trial['ng2'].iloc[0]),
                    'significant': float(aov_trial['p-unc'].iloc[0]) < 0.05
                }
            except Exception as e:
                group_results['anova_error'] = str(e)
            
            # Post-hoc pairwise comparisons - FIXED WARNING
            try:
                if len(group_data['trial_type'].unique()) > 2:
                    # Fixed: Use observed=True to avoid deprecation warning
                    subject_means = group_data.groupby(['subject', 'trial_type'], observed=True)['success_rate'].mean().reset_index()
                    subject_means['subject'] = subject_means['subject'].astype(str)
                    subject_means['trial_type'] = subject_means['trial_type'].astype('category')
                    
                    # Use the modern pairwise_tests function
                    posthoc = pg.pairwise_tests(data=subject_means, dv='success_rate', 
                                              within='trial_type', subject='subject', 
                                              padjust='bonf', effsize='hedges')
                    
                    group_results['posthoc_comparisons'] = {}
                    for _, row in posthoc.iterrows():
                        comparison = f"{row['A']}_vs_{row['B']}"
                        group_results['posthoc_comparisons'][comparison] = {
                            'mean_diff': float(row.get('mean(A)', 0) - row.get('mean(B)', 0)) if 'mean(A)' in row else 0.0,
                            't_stat': float(row['T']), 'p_corrected': float(row['p-corr']),
                            'cohens_d': float(row.get('hedges', 0.0)), 'significant': row['p-corr'] < 0.05,
                            'effect_size_interpretation': self._interpret_cohens_d(float(row.get('hedges', 0.0)))
                        }
            except Exception as e:
                group_results['posthoc_error'] = str(e)
            
            # Descriptive statistics
            try:
                group_results['mean_success_rates'] = {
                    trial: float(group_data[group_data['trial_type'] == trial]['success_rate'].mean())
                    for trial in group_data['trial_type'].unique()
                }
                group_results['descriptive_stats'] = {
                    'n_subjects': group_info['n_subjects'], 'n_observations': len(group_data),
                    'age_info': {'mean_age': group_info['mean_age'], 'age_range': group_info['age_range']}
                }
            except Exception as e:
                group_results['descriptive_error'] = str(e)
            
            results['group_analyses'][group_name] = group_results
        
        return results
    
    def _interpret_cohens_d(self, d):
        """Interpret Cohen's d effect size"""
        abs_d = abs(d)
        if abs_d < 0.2: return "negligible"
        elif abs_d < 0.5: return "small"
        elif abs_d < 0.8: return "medium"
        else: return "large"

    def run_regression_analysis(self, trial_type: str = 'invis', condition: str = 'max',
                               predictors: List[str] = None) -> Dict:
        """Run regression analysis"""
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
        
        available_predictors = [p for p in predictors if p in self.filtered_df.columns]
        target_col = f'{trial_type}_sr_{condition}_const'
        
        if target_col not in self.filtered_df.columns:
            raise ValueError(f"Target column {target_col} not found")
        
        valid_data = self.filtered_df[available_predictors + [target_col]].dropna()
        if len(valid_data) < 10:
            raise ValueError(f"Insufficient data: only {len(valid_data)} valid samples")
        
        X, y = valid_data[available_predictors], valid_data[target_col]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = Pipeline([('scaler', StandardScaler()), ('regressor', LinearRegression())])
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        return {
            'model': model,
            'metrics': {
                'r2': r2_score(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'n_samples': len(valid_data)
            },
            'feature_importances': dict(zip(available_predictors, 
                                          np.abs(model.named_steps['regressor'].coef_)))
        }
    
    def run_correlation_analysis(self) -> Dict:
        """Run correlation analysis between key variables"""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols)
        
        corr_data = self.filtered_df[key_cols].corr()
        
        significant_corrs = {}
        for i, col1 in enumerate(key_cols):
            for j, col2 in enumerate(key_cols):
                if i < j:
                    valid_data = self.filtered_df[[col1, col2]].dropna()
                    if len(valid_data) > 5:
                        r, p = pearsonr(valid_data[col1], valid_data[col2])
                        if p < 0.05:
                            significant_corrs[f'{col1}_vs_{col2}'] = {'r': r, 'p': p, 'n': len(valid_data)}
        
        return {'correlation_matrix': corr_data.to_dict(), 'significant_correlations': significant_corrs}

# ==============================================================================
# VISUALIZATION COMPONENTS - UNCHANGED (Too long to repeat)
# ==============================================================================

# The visualization classes remain the same - they don't contain the problematic groupby calls
# that were generating the warnings. The warnings were specifically from pingouin and statsmodels.

class PopulationVisualizer(BaseVisualizer):
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        super().__init__(config)
        self.metrics_df = metrics_df
        self.filtered_df = (metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold] 
                           if 'mot_noise' in metrics_df.columns else metrics_df)

    def _create_age_vs_metric_plot(self, metric_suffix: str, ylabel: str, title_suffix: str, 
                                  ylim: Tuple[float, float] = None) -> Path:
        """Generic function to create age vs metric plots"""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate shared axis limits
        metric_cols = [f'{trial}_{metric_suffix}_{condition}_const' for trial in trials for condition in conditions]
        available_cols = [col for col in metric_cols if col in self.filtered_df.columns]
        
        if available_cols and not ylim:
            all_data = []
            for col in available_cols:
                data = self.filtered_df[col].dropna()
                if not data.empty:
                    all_data.extend(data.values)
            
            if all_data:
                y_min, y_max = min(all_data), max(all_data)
                y_range = y_max - y_min
                y_padding = y_range * 0.05
                ylim = [y_min - y_padding, y_max + y_padding]
        
        # Calculate age limits
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min, age_max = age_data.min(), age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_{metric_suffix}_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel(ylabel)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                
                # Set consistent axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
                if ylim:
                    ax.set_ylim(ylim)
        
        plt.suptitle(f'Age vs {title_suffix} by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'age_vs_{metric_suffix}.png'
        return self.save_figure(fig, filename, 'population')

    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('sr', 'Success Rate', 'Success Rates', (-0.05, 1.05))
    
    def plot_age_vs_mean_stride_length(self) -> Path:
        """Plot age vs mean stride length by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('msl', 'Mean Stride Length', 'Mean Stride Length')
    
    def plot_age_vs_stride_variability(self) -> Path:
        """Plot age vs stride length variability by trial/condition with motor noise coloring"""
        return self._create_age_vs_metric_plot('sd', 'Stride Length Variability (SD)', 'Stride Length Variability')

    # [Rest of visualization methods remain the same for brevity]
    # ... (correlation_matrix, trial_comparison, anova_results, etc.)

# ==============================================================================
# MAIN ANALYSIS INTERFACE - UNCHANGED
# ==============================================================================

class MotorLearningAnalysis:
    """Main analysis interface that coordinates all components"""
    
    def __init__(self, config: AnalysisConfig = None):
        # Ensure we always have a config
        self.config = config or AnalysisConfig()
        self.data_manager: Optional[DataManager] = None
        self.metrics_df: Optional[pd.DataFrame] = None
        self.statistical_analyzer: Optional[StatisticalAnalyzer] = None
        self.population_visualizer: Optional[PopulationVisualizer] = None
        self.individual_visualizer: Optional['IndividualVisualizer'] = None
        self.results: Optional[Dict] = None
    
    def load_data(self, metadata_path: str, data_root_dir: str, 
                  force_reprocess: bool = False) -> 'MotorLearningAnalysis':
        """Load and process data"""
        self.data_manager = DataManager(metadata_path, data_root_dir, self.config, force_reprocess)
        return self
    
    def filter_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """
        Flexible filtering method that works both before and after metrics calculation.
        
        Pre-metrics filtering (filters raw subject data):
        - required_trial_types: List[str] - trials that must be present
        - min_age, max_age: float - age range filtering
        - max_target_size: float - maximum target size allowed
        
        Post-metrics filtering (filters calculated metrics):
        Any column name from the metrics DataFrame can be used with:
        - min_{column_name}: minimum value
        - max_{column_name}: maximum value
        - {column_name}: exact value or list of values
        - exclude_{column_name}: values to exclude
        
        Examples:
        .filter_data(required_trial_types=['vis1', 'invis'])  # Pre-metrics
        .filter_data(min_age=8, max_age=16)  # Pre or post-metrics
        .filter_data(min_mot_noise=0.1, max_mot_noise=0.3)  # Post-metrics
        .filter_data(min_vis1_sr_max_const=0.5)  # Post-metrics
        """
        
        if self.metrics_df is None:
            # Pre-metrics filtering - filter raw data
            return self._filter_raw_data(**kwargs)
        else:
            # Post-metrics filtering - filter calculated metrics
            return self._filter_metrics_data(**kwargs)
    
    def _filter_raw_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter raw subject data before metrics calculation"""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        # Extract parameters for raw data filtering
        filter_params = {}
        
        # Direct mapping for existing parameters
        raw_data_params = ['required_trial_types', 'min_age', 'max_age', 'max_target_size']
        for param in raw_data_params:
            if param in kwargs:
                filter_params[param] = kwargs[param]
        
        # Convert age parameters if present
        if 'min_age' in kwargs or 'max_age' in kwargs:
            filter_params.update({k: v for k, v in kwargs.items() if k in ['min_age', 'max_age']})
        
        # Apply filtering using existing DataManager method
        if filter_params:
            self.data_manager = self.data_manager.filter_subjects(**filter_params)
            print(f"Pre-metrics filtering applied. Remaining subjects: {len(self.data_manager.subjects)}")
        
        return self
    
    def _filter_metrics_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter calculated metrics data"""
        if self.metrics_df is None:
            raise ValueError("No metrics to filter. Call calculate_metrics() first.")
        
        original_count = len(self.metrics_df)
        filtered_df = self.metrics_df.copy()
        applied_filters = []
        
        for param, value in kwargs.items():
            if param in ['required_trial_types', 'max_target_size']:
                # Skip raw data parameters when filtering metrics
                continue
                
            # Handle min_* parameters
            if param.startswith('min_'):
                column_name = param[4:]  # Remove 'min_' prefix
                if column_name in filtered_df.columns:
                    mask = filtered_df[column_name] >= value
                    filtered_df = filtered_df[mask | filtered_df[column_name].isna()]
                    applied_filters.append(f"{column_name} >= {value}")
                else:
                    print(f"Warning: Column '{column_name}' not found for min filter")
            
            # Handle max_* parameters
            elif param.startswith('max_'):
                column_name = param[4:]  # Remove 'max_' prefix
                if column_name in filtered_df.columns:
                    mask = filtered_df[column_name] <= value
                    filtered_df = filtered_df[mask | filtered_df[column_name].isna()]
                    applied_filters.append(f"{column_name} <= {value}")
                else:
                    print(f"Warning: Column '{column_name}' not found for max filter")
            
            # Handle exclude_* parameters
            elif param.startswith('exclude_'):
                column_name = param[8:]  # Remove 'exclude_' prefix
                if column_name in filtered_df.columns:
                    if isinstance(value, (list, tuple)):
                        mask = ~filtered_df[column_name].isin(value)
                    else:
                        mask = filtered_df[column_name] != value
                    filtered_df = filtered_df[mask | filtered_df[column_name].isna()]
                    applied_filters.append(f"{column_name} not in {value}")
                else:
                    print(f"Warning: Column '{column_name}' not found for exclude filter")
            
            # Handle direct column filtering
            elif param in filtered_df.columns:
                if isinstance(value, (list, tuple)):
                    mask = filtered_df[param].isin(value)
                    applied_filters.append(f"{param} in {value}")
                else:
                    mask = filtered_df[param] == value
                    applied_filters.append(f"{param} == {value}")
                filtered_df = filtered_df[mask | filtered_df[param].isna()]
            else:
                print(f"Warning: Unknown parameter '{param}' - not found in metrics columns")
        
        # Update metrics and reinitialize analyzers
        self.metrics_df = filtered_df
        if not filtered_df.empty:
            self._initialize_analyzers()
        
        filtered_count = len(filtered_df)
        excluded_count = original_count - filtered_count
        
        if applied_filters:
            print(f"Post-metrics filtering applied:")
            for filter_desc in applied_filters:
                print(f"  - {filter_desc}")
            print(f"  Original subjects: {original_count}")
            print(f"  Excluded subjects: {excluded_count}")
            print(f"  Remaining subjects: {filtered_count}")
        
        return self
    
    def get_available_filters(self) -> Dict[str, List[str]]:
        """Get available filter parameters based on current state"""
        available = {
            'pre_metrics': [
                'required_trial_types', 'min_age', 'max_age', 'max_target_size'
            ],
            'post_metrics': []
        }
        
        if self.metrics_df is not None:
            columns = list(self.metrics_df.columns)
            available['post_metrics'] = columns
            available['post_metrics_examples'] = [
                f"min_{col}" for col in columns[:5]
            ] + [f"max_{col}" for col in columns[:5]] + [
                f"exclude_{col}" for col in columns[:3]
            ]
        
        return available
    
    def describe_data(self) -> Dict:
        """Get summary statistics about current data"""
        info = {}
        
        if self.data_manager:
            info['raw_subjects'] = len(self.data_manager.subjects)
            if self.data_manager.metadata is not None:
                age_data = self.data_manager.metadata.get('age', pd.Series([]))
                if not age_data.empty:
                    info['age_range'] = [float(age_data.min()), float(age_data.max())]
        
        if self.metrics_df is not None:
            info['metrics_subjects'] = len(self.metrics_df)
            info['metrics_columns'] = list(self.metrics_df.columns)
            
            # Key statistics
            numeric_cols = self.metrics_df.select_dtypes(include=[np.number]).columns
            stats = {}
            for col in numeric_cols[:10]:  # First 10 numeric columns
                col_data = self.metrics_df[col].dropna()
                if not col_data.empty:
                    stats[col] = {
                        'mean': float(col_data.mean()),
                        'std': float(col_data.std()),
                        'min': float(col_data.min()),
                        'max': float(col_data.max()),
                        'count': int(len(col_data))
                    }
            info['key_statistics'] = stats
        
        return info
    
    def export_metrics(self, format: str = 'all', custom_path: str = None) -> List[str]:
            """Export metrics in specified format(s) using config paths"""
            if self.metrics_df is None:
                raise ValueError("No metrics to export. Call calculate_metrics() first.")
            
            exported_files = []
            base_name = custom_path or f"metrics_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            if format in ['csv', 'all']:
                # Use config.exportsdir
                csv_path = self.config.exportsdir / f"{base_name}.csv"
                self.metrics_df.to_csv(csv_path, index=False)
                exported_files.append(str(csv_path))
            
            if format in ['excel', 'all']:
                # Use config.exportsdir
                excel_path = self.config.exportsdir / f"{base_name}.xlsx"
                self.metrics_df.to_excel(excel_path, index=False, engine='openpyxl')
                exported_files.append(str(excel_path))
            
            if format in ['pickle', 'all']:
                # Use config.exportsdir
                pickle_path = self.config.exportsdir / f"{base_name}.pkl"
                self.metrics_df.to_pickle(pickle_path)
                exported_files.append(str(pickle_path))
            
            print(f"Exported metrics to: {exported_files}")
            return exported_files
    
    def calculate_metrics(self, use_cached: bool = True) -> 'MotorLearningAnalysis':
        """Calculate metrics for all subjects"""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        # Try to load cached metrics first if requested
        if use_cached:
            cached_metrics = self.data_manager.load_metrics()
            if cached_metrics is not None:
                self.metrics_df = cached_metrics
                self._initialize_analyzers()
                return self
        
        # Calculate new metrics
        self.metrics_df = self.data_manager.calculate_metrics()
        self._initialize_analyzers()
        return self
    
    def _initialize_analyzers(self):
        """Initialize analyzers with metrics data"""
        self.statistical_analyzer = StatisticalAnalyzer(self.metrics_df, self.config)
        self.population_visualizer = PopulationVisualizer(self.metrics_df, self.config)
        # self.individual_visualizer = IndividualVisualizer(self.data_manager, self.config)
    
    def get_column_stats(self, column: str) -> Dict:
        """Get statistics about any column in the dataset"""
        if self.metrics_df is None:
            return {'error': 'No metrics data available'}
        
        if column not in self.metrics_df.columns:
            return {'error': f'Column {column} not found in metrics'}
        
        col_data = self.metrics_df[column].dropna()
        if col_data.empty:
            return {'error': f'No valid data in column {column}'}
        
        # Basic stats for all data types
        stats = {
            'column': column,
            'n_subjects_with_data': len(col_data),
            'n_subjects_missing_data': self.metrics_df[column].isna().sum(),
            'data_type': str(col_data.dtype)
        }
        
        # Additional stats for numeric data
        if pd.api.types.is_numeric_dtype(col_data):
            stats.update({
                'mean': float(col_data.mean()),
                'std': float(col_data.std()),
                'min': float(col_data.min()),
                'max': float(col_data.max()),
                'median': float(col_data.median()),
                'q25': float(col_data.quantile(0.25)),
                'q75': float(col_data.quantile(0.75))
            })
            
            # Add threshold info for motor noise specifically
            if column == 'mot_noise':
                threshold = self.config.motor_noise_threshold
                stats.update({
                    'threshold': threshold,
                    'n_above_threshold': int((col_data > threshold).sum()),
                    'n_below_threshold': int((col_data <= threshold).sum())
                })
        else:
            # Stats for categorical data
            value_counts = col_data.value_counts()
            stats.update({
                'unique_values': len(value_counts),
                'most_common': value_counts.head().to_dict()
            })
        
        return stats
    
    def run_analysis(self, include_visualizations: bool = True) -> 'MotorLearningAnalysis':
        """Run comprehensive analysis and store results"""
        if self.metrics_df is None:
            raise ValueError("Metrics not calculated. Call calculate_metrics() first.")
        
        self.results = {
            'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'n_subjects': len(self.metrics_df),
            'analyses': {}
        }
        
        # Statistical analyses - consolidated error handling
        analyses = [
            ('regression', lambda: self.statistical_analyzer.run_regression_analysis()),
            ('anova_success_rates', lambda: self.statistical_analyzer.run_repeated_measures_anova()),
            ('anova_mean_stride_length', lambda: self.statistical_analyzer.run_repeated_measures_anova_msl()),
            ('anova_stride_variability', lambda: self.statistical_analyzer.run_repeated_measures_anova_sd()),
            ('correlations', lambda: self.statistical_analyzer.run_correlation_analysis()),
            ('age_stratified_anova', lambda: self.statistical_analyzer.run_age_stratified_anova())
        ]
        
        for name, func in analyses:
            try:
                self.results['analyses'][name] = func()
            except Exception as e:
                self.results['analyses'][name] = {'error': str(e)}
        
        # Visualizations
        if include_visualizations and self.population_visualizer:
            self.results['visualizations'] = {'population': []}
            
            # Population plots - consolidated error handling
            plot_functions = [
                self.population_visualizer.plot_age_vs_success_rates,
                self.population_visualizer.plot_age_vs_mean_stride_length,
                self.population_visualizer.plot_age_vs_stride_variability,
            ]
            
            for plot_func in plot_functions:
                try:
                    plot_path = plot_func()
                    if plot_path:
                        self.results['visualizations']['population'].append(str(plot_path))
                except Exception as e:
                    print(f"Failed to create plot {plot_func.__name__}: {e}")
        
        # Generate report
        self._generate_report()
        return self
    
    def get_results(self) -> Dict:
        """Get analysis results"""
        return self.results
    
    def _generate_report(self):
        """Generate analysis report"""
        if not self.results:
            return
        
        report_path = self.config.reportsdir / f"analysis_report_{self.results['timestamp']}.json"
        
        def make_serializable(obj):
            if isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, Path):
                return str(obj)
            elif pd.isna(obj):
                return None
            elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
                return str(type(obj).__name__)
            return obj
        
        serializable_results = json.loads(json.dumps(self.results, default=make_serializable))
        
        with open(report_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"Report saved to: {report_path}")

# ==============================================================================
# CONVENIENCE FUNCTIONS
# ==============================================================================

def run_motor_learning_analysis(metadata_path: str, data_root_dir: str,
                               output_dir: str = 'motor_learning_output',
                               required_trials: List[str] = None,
                               apply_motor_noise_filter: bool = True,
                               **filter_kwargs) -> MotorLearningAnalysis:
    """
    Convenience function to run complete analysis with flexible filtering.
    
    Args:
        metadata_path: Path to metadata CSV
        data_root_dir: Path to data directory
        output_dir: Output directory path
        required_trials: Required trial types. If None, defaults to all trials ['vis1', 'invis', 'vis2']
        apply_motor_noise_filter: Whether to automatically apply motor noise filtering (default: True)
        **filter_kwargs: Any additional filtering parameters
    
    Examples:
        # Basic usage - requires all trials by default ['vis1', 'invis', 'vis2']
        analysis = run_motor_learning_analysis(metadata_path, data_root_dir)
        
        # Require only specific trials
        analysis = run_motor_learning_analysis(
            metadata_path, data_root_dir,
            required_trials=['vis1', 'invis']  # Only these two
        )
        
        # Require no specific trials (accept subjects with any trial data)
        analysis = run_motor_learning_analysis(
            metadata_path, data_root_dir,
            required_trials=[]  # Empty list = no requirements
        )
        
        # Disable automatic motor noise filtering
        analysis = run_motor_learning_analysis(
            metadata_path, data_root_dir,
            apply_motor_noise_filter=False
        )
        
        # Custom filtering with overridden motor noise threshold
        analysis = run_motor_learning_analysis(
            metadata_path, data_root_dir,
            max_mot_noise=0.25  # This overrides the config default
        )
    """
    config = AnalysisConfig(base_output_dir=Path(output_dir))
    
    # Set default required trials to all trial types if not specified
    if required_trials is None:
        required_trials = ['vis1', 'invis', 'vis2']
        print(f"🔧 Default trial requirement: {required_trials}")
    elif required_trials == []:
        print("🔧 No trial requirements - accepting subjects with any trial data")
    else:
        print(f"🔧 Custom trial requirement: {required_trials}")
    
    # Separate pre-metrics and post-metrics filters
    pre_metrics_filters = {}
    post_metrics_filters = {}
    
    # Pre-metrics filter parameters
    pre_metrics_params = ['required_trial_types', 'min_age', 'max_age', 'max_target_size']
    
    # Add required_trials to pre-metrics filters (only if not empty list)
    if required_trials:  # This will be False for empty list [], True for None or populated list
        pre_metrics_filters['required_trial_types'] = required_trials
    
    # Separate filter parameters
    for key, value in filter_kwargs.items():
        if key in pre_metrics_params:
            pre_metrics_filters[key] = value
        else:
            post_metrics_filters[key] = value
    
    # Add automatic motor noise filtering if enabled and not overridden
    if apply_motor_noise_filter and 'max_mot_noise' not in post_metrics_filters:
        post_metrics_filters['max_mot_noise'] = config.motor_noise_threshold
        print(f"🔧 Automatic motor noise filtering enabled (threshold: {config.motor_noise_threshold})")
    
    # Build analysis pipeline
    analysis = MotorLearningAnalysis(config).load_data(metadata_path, data_root_dir)
    
    # Apply pre-metrics filters
    if pre_metrics_filters:
        analysis = analysis.filter_data(**pre_metrics_filters)
    
    # Calculate metrics
    analysis = analysis.calculate_metrics()
    
    # Apply post-metrics filters (including automatic motor noise filter)
    if post_metrics_filters:
        analysis = analysis.filter_data(**post_metrics_filters)
    
    return analysis.run_analysis()


def create_analysis_pipeline(metadata_path: str, data_root_dir: str,
                           output_dir: str = 'motor_learning_output') -> MotorLearningAnalysis:
    """
    Create analysis pipeline for step-by-step filtering and analysis.
    
    Returns an analysis object that you can chain filters on:
    
    analysis = (create_analysis_pipeline(metadata_path, data_root_dir)
                .filter_data(required_trial_types=['vis1', 'invis'])
                .filter_data(min_age=8, max_age=16)
                .calculate_metrics()
                .filter_data(max_mot_noise=0.3)
                .filter_data(min_vis1_sr_max_const=0.5)
                .run_analysis())
    """
    config = AnalysisConfig(base_output_dir=Path(output_dir))
    return MotorLearningAnalysis(config).load_data(metadata_path, data_root_dir)


def main():
    """Example usage"""
    data_root_dir = 'muh_data/'
    metadata_path = 'muh_metadata.csv'
    
    print("Motor Learning Analysis Pipeline - Optimized Version")
    print("=" * 60)
    
    # Quick analysis
    analysis = run_motor_learning_analysis(metadata_path, data_root_dir)
    
    # Access components
    data_manager = analysis.data_manager
    metrics_df = analysis.metrics_df
    stats = analysis.statistical_analyzer
    
    # Run specific analyses
    regression_results = stats.run_regression_analysis()
    anova_results = stats.run_repeated_measures_anova()
    
    # Print summary
    print(f"\nDataset Summary:")
    print(f"  Total subjects: {len(metrics_df)}")
    print(f"  Age range: {metrics_df['age'].min():.1f} - {metrics_df['age'].max():.1f} years")
    
    print(f"\nRegression R²: {regression_results['metrics']['r2']:.3f}")
    print(f"ANOVA trial effect p-value: {anova_results.get('trial_type_effect', {}).get('p_value', 'N/A')}")
    
    print(f"\nAll outputs saved to: {analysis.config.base_output_dir}")
    return analysis


if __name__ == "__main__":
    main()

Motor Learning Analysis Pipeline - Optimized Version
🔧 Default trial requirement: ['vis1', 'invis', 'vis2']
🔧 Automatic motor noise filtering enabled (threshold: 0.3)
✓ Loaded 110 subjects from cache
Pre-metrics filtering applied. Remaining subjects: 66
✓ Loaded metrics for 66 subjects from motor_learning_output\processed_data\calculated_metrics.pkl
Post-metrics filtering applied:
  - mot_noise <= 0.3
  Original subjects: 66
  Excluded subjects: 1
  Remaining subjects: 65
Figure saved to: motor_learning_output\figures\population_plots\age_vs_sr.png
Figure saved to: motor_learning_output\figures\population_plots\age_vs_msl.png
Figure saved to: motor_learning_output\figures\population_plots\age_vs_sd.png
Report saved to: motor_learning_output\reports\analysis_report_20250723_132233.json

Dataset Summary:
  Total subjects: 65
  Age range: 7.2 - 17.9 years

Regression R²: -0.215
ANOVA trial effect p-value: 0.0011120909891778382

All outputs saved to: motor_learning_output
